# BM25: Sparse Keyword Retrieval

**BM25** ranks documents by how well their keywords overlap with the query. For each query token it combines:

- **Term frequency (TF)** — more occurrences of a token score higher, with diminishing returns.
- **Inverse document frequency (IDF)** — rare tokens are worth more than common ones.
- **Length normalization** — long chunks don't win just by containing more words.

There is no neural network involved: indexing and querying are fast and fully interpretable, but only tokens that *literally appear* in a chunk can match.

In this notebook we index the Google 10-K filing, run keyword queries, inspect the raw BM25 scores, and probe where the approach breaks down.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from rag.data_ingestion import load_documents, chunk_documents
from rag.retrievers import BM25Retriever

PDF_PATH = ROOT / "data" / "google_10K.pdf"

documents = load_documents(PDF_PATH)
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")

Loaded 107 pages -> 433 chunks


## Indexing

`BM25Retriever` tokenizes every chunk with a plain whitespace `split()` and feeds the token lists to `BM25Okapi`, which precomputes the term statistics.

In [2]:
retriever = BM25Retriever()
retriever.add_documents(chunks)

print(f"Indexed {len(retriever.documents)} chunks")

Indexed 433 chunks


## Retrieval

A keyword-heavy query is BM25's home turf: the query tokens appear verbatim in the financial statements.

In [3]:
query = "total revenues 2024 2023"

results = retriever.retrieve(query, top_k=5)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    preview = doc.page_content[:150].replace("\n", " ")
    print(f"{i}. [page {doc.metadata['page']}] {preview}...\n")

Query: total revenues 2024 2023

1. [page 66] vices 34,688  40,340  48,030  Google Services total 272,543  304,930  342,721  Google Cloud 33,088  43,229  58,705  Other Bets 1,527  1,648  1,537  He...

2. [page 66] Table of Contents Alphabet Inc. In December 2023, the FASB issued ASU 2023-09 "Income Taxes (Topics 740): Improvements to Income Tax Disclosures" whic...

3. [page 39] he year ended December 31, 2025. • As of December 31, 2025, we had 190,820 employees. We are monitoring ongoing developments surrounding international...

4. [page 41] Table of Contents Alphabet Inc. Cost of Revenues The following table presents cost of revenues, including TAC (in millions, except percentages):   Yea...

5. [page 34] Table of Contents Alphabet Inc. We use certain metrics to track how well traffic across various properties is monetized as it relates to our advertisi...



## Inspecting the scores

`retrieve()` returns documents but hides the scores. To see them, call the underlying `BM25Okapi` index directly.

In [4]:
query = "total revenues 2024"
tokens = query.split()
scores = retriever.bm25.get_scores(tokens)

top_indices = np.argsort(scores)[::-1][:10]

print(f"Query tokens: {tokens}\n")
print(f"{'Rank':<6} {'Score':<10} {'Page':<6} Preview")
print("-" * 70)
for rank, idx in enumerate(top_indices, start=1):
    doc = retriever.documents[idx]
    preview = doc.page_content[:40].replace("\n", " ")
    print(f"{rank:<6} {scores[idx]:<10.2f} {doc.metadata['page']:<6} {preview}...")

Query tokens: ['total', 'revenues', '2024']

Rank   Score      Page   Preview
----------------------------------------------------------------------
1      6.95       66     vices 34,688  40,340  48,030  Google Ser...
2      6.53       39     he year ended December 31, 2025. • As of...
3      6.38       41     Table of Contents Alphabet Inc. Cost of ...
4      6.34       34     Table of Contents Alphabet Inc. We use c...
5      5.84       66     Table of Contents Alphabet Inc. In Decem...
6      5.81       44      example, our data center construction p...
7      5.54       40     Table of Contents Alphabet Inc. YouTube ...
8      5.51       41      revenues were substantially consistent ...
9      5.25       71     ative net gains (losses), calculated as ...
10     5.09       34     ssion is defined as impression-based and...


## Limitation 1: naive tokenization

Whitespace splitting keeps punctuation attached to tokens and preserves case, so `"income"` and `"income)"` are different terms — and so are `"Revenues"` and `"revenues"`.

In [5]:
sample = "Operating income (loss) of $50.2B in 2024, up 15% YoY."
print(f"Text:   {sample}")
print(f"Tokens: {sample.split()}\n")

ids_lower = [r.metadata["doc_id"] for r in retriever.retrieve("operating revenues", top_k=5)]
ids_title = [r.metadata["doc_id"] for r in retriever.retrieve("Operating Revenues", top_k=5)]

print(f"'operating revenues' -> {ids_lower}")
print(f"'Operating Revenues' -> {ids_title}")
if ids_lower == ids_title:
    print("\nSame results for this query (both casings happen to appear in the text).")
else:
    print("\nDifferent results: matching is case-sensitive.")

Text:   Operating income (loss) of $50.2B in 2024, up 15% YoY.
Tokens: ['Operating', 'income', '(loss)', 'of', '$50.2B', 'in', '2024,', 'up', '15%', 'YoY.']

'operating revenues' -> ['chunk_71', 'chunk_173', 'chunk_198', 'chunk_72', 'chunk_200']
'Operating Revenues' -> ['chunk_179', 'chunk_325', 'chunk_326', 'chunk_320', 'chunk_386']

Different results: matching is case-sensitive.


## Limitation 2: no semantics

BM25 has no notion of meaning — it can only reward token overlap. A conversational paraphrase like *"how much money did the company earn"* still produces high raw scores, but for the wrong reason: the points come from incidental words (`how`, `did`, `money`, `company`), so the top chunks are off-topic. Raw scores are not comparable across queries; what matters is *which* chunks come back.

In [6]:
queries = [
    ("total revenues 2024", "Exact keywords"),
    ("how much money did the company earn", "Semantic paraphrase"),
]

for query, query_type in queries:
    print(f"{query_type}: {query!r}")
    for i, doc in enumerate(retriever.retrieve(query, top_k=3), start=1):
        preview = doc.page_content[:90].replace("\n", " ")
        print(f"  {i}. [page {doc.metadata['page']}] {preview}...")
    print()

print("The paraphrase's hits match filler words, not the revenue tables.")

Exact keywords: 'total revenues 2024'
  1. [page 66] vices 34,688  40,340  48,030  Google Services total 272,543  304,930  342,721  Google Clou...
  2. [page 39] he year ended December 31, 2025. • As of December 31, 2025, we had 190,820 employees. We a...
  3. [page 41] Table of Contents Alphabet Inc. Cost of Revenues The following table presents cost of reve...

Semantic paraphrase: 'how much money did the company earn'
  1. [page 60] Table of Contents Alphabet Inc. • consumer subscriptions, which primarily include revenues...
  2. [page 62]  the concentration of our credit risk exposure by performing ongoing evaluations to determ...
  3. [page 43] n Cash, Cash Equivalents, and Marketable Securities As of December 31, 2025, we had $126.8...

The paraphrase's hits match filler words, not the revenue tables.


## Takeaways

- BM25 is fast, cheap, and excellent when queries share vocabulary with the documents (financial line items, tickers, exact phrases).
- The whitespace tokenizer makes matching sensitive to punctuation and casing.
- Paraphrased or conversational queries can still score high on incidental words, but the retrieved chunks miss the intent — BM25 matches tokens, not meaning.

The next notebook, `02_dense.ipynb`, addresses the semantic gap with embedding-based retrieval.